# PhysAR-IDS -- NB3d (v6): do ransomware and reverseshell give way to the white-box attack with more search?

**Why.** NB3c's white-box (tree-greedy) attack evaded backdoor, Bruteforce and FDI at every budget, but gave 0.000 on ransomware and reverseshell up to 0.3 sigma and about 0.02 / 0.001 at 0.5 sigma, then 1.000 at 1.0 sigma. The run log shows that zero is **not converged**: it used 25 greedy rounds, 29-38% of rows hit the round limit while their margin was still improving (only 0.5-5% had no helpful move left), and at 0.5 sigma the evasion was still creeping up in the last rounds (seed 42: 0.611 at round 15, 0.631 at round 25; ransomware 0.106). So "0 up to 0.5 sigma" is a lower bound from an unfinished search, not a threshold. This notebook continues the same attack on the two hard classes for up to **150 rounds** and records the trajectory.

**Design.** Exactly NB3c's attack, model, budgets, range clip and oracle protocol. Changes: only ransomware and reverseshell rows (3,000 of each, drawn once with a fixed generator, identical for all seeds and settings), `MAX_ITERS = 150`, and a per-round trace of (i) the share evaded and (ii) the median margin gap of the rows not yet evaded (gap > 0 means benign wins; a gap that keeps closing means the search is still making progress). Budgets: 0.3 and 0.5 sigma (edit `SETTINGS_TO_RUN` to add 0.7). The first 25 rounds of this run are directly comparable with NB3c's full-set numbers (a check below).

**How to read it.**
* Evasion rises with rounds and reaches a high level -> the hard classes are evadable at that budget; the threshold from NB3c was an artefact of the round limit.
* Evasion stays ~0 while the median gap has flattened (stalled rows, no helpful single move) -> no evasion found by this search at that budget. That is still a lower bound (greedy, one feature per round), but a much stronger statement than NB3c's.
* Evasion stays ~0 but the gap is still closing at round 150 -> still unconverged; more rounds or a different search are needed.

## 0. Setup

In [ ]:
import os, sys, time, json, hashlib, warnings
import numpy as np, pandas as pd, joblib
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False
K_CAND = 12            # as NB3c
MAX_ITERS = 150        # NB3c used 25: this notebook asks whether the hard classes give way with more rounds
CHUNK = 20000; BLOCK = 300_000       # candidate rows scored per model call (NB3c used 1,000,000; smaller to leave GPU memory free)
N_PER_CLASS = 3000     # ransomware and reverseshell rows searched (same rows for every seed and setting)
CHECKPOINTS = [25, 50, 100, 150]
SETTINGS = {'cyber_0.3': ('cyber', 0.3), 'cyber_0.5': ('cyber', 0.5), 'cyber_0.7': ('cyber', 0.7)}
SETTINGS_TO_RUN = ['cyber_0.3', 'cyber_0.5']       # add 'cyber_0.7' to locate the threshold more finely

os.environ['PYTHONHASHSEED'] = '42'
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
print(f'xgboost {xgb.__version__} | search backend: {"cpu(hist)" if _cpu else "gpu(cuda,hist)"} | working directory: {os.getcwd()}')

## 1. Load v6 data and the NB2 tuned configuration

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; ALL_IDX = np.arange(N_FEAT)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
atk_te = y_test != 0; X_atk_test, y_atk_test = X_test[atk_te], y_test[atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | cyber (controllable in the main settings) {len(CYBER_IDX)} | evaluation attack rows {X_atk_test.shape} | class counts {np.bincount(y_atk_test)[1:]}')
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. The attack (as NB3c) with a per-round trace

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access). Scores in blocks and, if the GPU runs out of memory, halves the block and retries."""
    def __init__(self, clf): self.b = clf.get_booster(); self.bs = BLOCK
    def _pred(self, X):
        while True:
            try:
                return [np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + self.bs], dtype=np.float32), predict_type='margin', validate_features=False)) for s in range(0, len(X), self.bs)]
            except xgb.core.XGBoostError as e:
                if 'out of memory' not in str(e).lower() or self.bs <= 20000: raise
                self.bs = max(20000, self.bs // 2); print(f'      GPU out of memory -> scoring in blocks of {self.bs:,} rows', flush=True)
    def __call__(self, X):
        out = self._pred(X); return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox_trace(margin, X0, y0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    """as NB3c's whitebox, plus a trace: per round, the evaded share per class (classes 4, 5) and the median margin gap of the rows not yet evaded."""
    x = X0.copy(); g = gap_of(margin(x)); active = g <= 0; stalled = np.zeros(len(x), bool); trace = []; t_start = time.time()
    active = ~(g > 0); init_ev = g > 0
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); ev = g > 0
        trace.append({'round': it + 1, 'active': int(len(act)), 'evaded_c4': float(ev[y0 == 4].mean()), 'evaded_c5': float(ev[y0 == 5].mean()),
                      'median_gap_unevaded': float(np.median(g[~ev])) if (~ev).any() else float('nan'), 'seconds': time.time() - t0})
        if verbose and ((it + 1) % 10 == 0 or it == 0): print(f'      round {it + 1:>3}: active {len(act):>5} | evaded ransomware {trace[-1]["evaded_c4"]:.3f} reverseshell {trace[-1]["evaded_c5"]:.3f} | median gap of unevaded {trace[-1]["median_gap_unevaded"]:+.3f} | {time.time() - t0:4.1f}s', flush=True)
    return x, g, init_ev, stalled, trace
def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):      # used by the self-test
    return whitebox_trace(margin, X0, np.zeros(len(X0), dtype=np.int64), feats, TH, B, K, max_iters, chunk, verbose)
print('white-box attack defined')

In [ ]:
def selftest(clf, margin, TH, seed=0):
    rng = np.random.RandomState(seed); rows = rng.choice(len(X_atk_test), 60, replace=False); X0 = X_atk_test[rows]; feats = rng.choice(CYBER_IDX, 12, replace=False)
    g0 = gap_of(margin(X0)); worst = 0.0
    for B in [0.1, 0.5, None]:
        bg, bj, bv, _ = one_step(margin, X0.copy(), X0, g0, feats, TH, B, K=10 ** 6)
        for i in range(len(rows)):
            best = g0[i]
            for j in feats:
                T = TH.get(int(j))
                if T is None: continue
                a, b = interval(X0[i:i + 1, j], int(j), B); a, b = float(a[0]), float(b[0]); inner = T[(T > a) & (T < b)]
                pts = np.concatenate([[a, b], (inner[:-1] + inner[1:]) / 2 if len(inner) > 1 else [], inner, np.nextafter(inner, np.float32(-np.inf))]).astype(np.float32)
                pts = pts[(pts >= a) & (pts <= b)]
                Xc = np.repeat(X0[i:i + 1], len(pts), 0); Xc[:, j] = pts; best = max(best, gap_of(margin(Xc)).max())
            worst = max(worst, best - bg[i])
            assert bg[i] >= best - 1e-4, f'one_step missed a better move (row {i}, budget {B}): {bg[i]} < {best}'
    # constraints after a short attack
    sub = X_atk_test[rng.choice(len(X_atk_test), 400, replace=False)]
    for B in [0.1, 0.5]:
        xa, ga, ie, st, lg = whitebox(margin, sub, CYBER_IDX, TH, B, max_iters=4, verbose=False)
        d = np.abs(xa - sub); assert d[:, CYBER_IDX].max() <= B + 1e-6, 'L-inf budget violated'
        other = np.setdiff1d(np.arange(N_FEAT), CYBER_IDX); assert d[:, other].max() == 0, 'a non-controllable feature changed'
        lo_ok = (xa >= np.minimum(FEAT_LO, sub) - 1e-6) & (xa <= np.maximum(FEAT_HI, sub) + 1e-6); assert lo_ok.all(), 'left the training range'
        assert (ga >= gap_of(margin(sub)) - 1e-9).all(), 'margin gap decreased'
    print(f'self-test passed (largest shortfall of one_step vs full enumeration: {max(worst, 0):.2e}); budget, range, non-controllable features and monotone margin all hold')

## 3. Rows searched
3,000 ransomware and 3,000 reverseshell test rows, drawn once (seeded generator 0), so every seed and budget sees the same rows and the comparison across seeds is paired. The rows come from the same 81,718 evaluation rows as NB3c; they are sampled at random, not by burst.

In [ ]:
rng0 = np.random.RandomState(0); pick = []
for c in (4, 5):
    idx = np.flatnonzero(y_atk_test == c); pick.append(np.sort(rng0.choice(idx, N_PER_CLASS, replace=False)))
pick = np.concatenate(pick); X_hard, y_hard = X_atk_test[pick], y_atk_test[pick]
print(f'rows searched: {len(X_hard)} (ransomware {int((y_hard == 4).sum())}, reverseshell {int((y_hard == 5).sum())}) | rounds {MAX_ITERS} | settings {SETTINGS_TO_RUN}')

## 4. Per-seed run (resumable per seed and setting)

In [ ]:
R = {}; INFO = {}; TR = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'attack1x_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = repr((best_params, K_CAND, MAX_ITERS, N_PER_CLASS, seed, X_hard.shape, float(X_hard[:5].sum()), 'nb3d-v1'))
    R[seed] = {}; TR[seed] = {}; INFO[seed] = {}
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed], TR[seed], INFO[seed] = saved['R'], saved['TR'], saved['info']
    todo = [s for s in SETTINGS_TO_RUN if s not in R[seed]]
    if not todo: print(f'seed {seed}: all settings found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}: {len(todo)} setting(s) to run', flush=True)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X_train, y_train, verbose=False)
    tmp = f'_tmp_xgb_seed{seed}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp)
    margin = Margin(clf); TH = get_thresholds(clf)
    if 'clean_weighted_f1' not in INFO[seed]: INFO[seed] = {'clean_weighted_f1': float(f1_score(y_test, cpu.predict(X_test), average='weighted'))}
    if seed == SEEDS[0] and not R[seed]: selftest(clf, margin, TH)
    for sname in todo:
        grp, B = SETTINGS[sname]; ts = time.time(); print(f'  [{sname}] searching {len(X_hard)} rows for up to {MAX_ITERS} rounds ...', flush=True)
        xa, ga, ie, st, tr = whitebox_trace(margin, X_hard, y_hard, CYBER_IDX, TH, B)
        pred_cpu = cpu.predict(xa); ev = pred_cpu == 0; d = np.abs(xa - X_hard); changed = (d > 0).sum(1)
        at = lambda r, k: tr[min(r, len(tr)) - 1][k] if tr else float('nan')          # the trace stops early when no row is active; the last value holds afterwards
        R[seed][sname] = {'final_evasion': np.array([ev[y_hard == 4].mean(), ev[y_hard == 5].mean()]), 'evasion_at': {r: np.array([at(r, 'evaded_c4'), at(r, 'evaded_c5')]) for r in CHECKPOINTS},
                          'clean_benign': np.array([ie[y_hard == 4].mean(), ie[y_hard == 5].mean()]), 'median_gap_at': {r: at(r, 'median_gap_unevaded') for r in CHECKPOINTS},
                          'features_changed_evaded': float(np.median(changed[ev])) if ev.any() else float('nan'), 'max_abs_change_max': float(d.max()),
                          'rows_stalled': float((st & ~ev).mean()), 'rows_round_limit': float((~st & ~ev).mean()), 'rounds_run': len(tr), 'search_time_s': float(time.time() - ts),
                          'search_vs_cpu_disagree': float(((ga > 0) != ev).mean())}
        TR[seed][sname] = tr; r_ = R[seed][sname]
        print(f'  [{sname}] final evasion ransomware {r_["final_evasion"][0]:.3f} reverseshell {r_["final_evasion"][1]:.3f} | at round 25: {np.round(r_["evasion_at"][25], 3).tolist()} | stalled {r_["rows_stalled"]:.3f} still-improving-at-limit {r_["rows_round_limit"]:.3f} | {(time.time() - ts) / 60:.1f} min', flush=True)
        joblib.dump({'fp': cfg_fp, 'R': R[seed], 'TR': TR[seed], 'info': INFO[seed]}, pf)
    print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 5. Results: evasion of the hard classes against the number of rounds (mean +/- std over seeds)

In [ ]:
cls = ['ransomware', 'reverseshell']
def have(s_): return [s for s in SEEDS if s_ in R[s]]
rows = []
for sname in SETTINGS_TO_RUN:
    if not have(sname): continue
    n = len(have(sname))
    for k, c in enumerate(cls):
        row = {'setting': sname, 'class': c, 'seeds': n}
        for r in CHECKPOINTS:
            v = np.array([R[s][sname]['evasion_at'][r][k] for s in have(sname)]); row[f'round {r}'] = f'{v.mean():.3f} +/- {(v.std(ddof=1) if n > 1 else 0):.3f}'
        fv = np.array([R[s][sname]['final_evasion'][k] for s in have(sname)]); row['final (CPU oracle)'] = f'{fv.mean():.3f} +/- {(fv.std(ddof=1) if n > 1 else 0):.3f}'
        row['stalled'] = f'{np.mean([R[s][sname]["rows_stalled"] for s in have(sname)]):.3f}'; row['still improving at limit'] = f'{np.mean([R[s][sname]["rows_round_limit"] for s in have(sname)]):.3f}'
        row['median gap of unevaded @25 -> @150'] = f'{np.mean([R[s][sname]["median_gap_at"][25] for s in have(sname)]):+.2f} -> {np.mean([R[s][sname]["median_gap_at"][150] for s in have(sname)]):+.2f}'
        rows.append(row)
conv_table = pd.DataFrame(rows); pd.set_option('display.width', 320); print(conv_table.to_string(index=False))
print('\nPer-seed final evasion [ransomware, reverseshell]:')
for sname in SETTINGS_TO_RUN:
    print(f'  {sname}:', {s: np.round(R[s][sname]['final_evasion'], 3).tolist() for s in have(sname)})
print('\nSearch/oracle disagreement (max over runs):', max(R[s][k]['search_vs_cpu_disagree'] for s in SEEDS for k in R[s]))

## 6. Consistency with NB3c (round 25 on the sample vs the full set)

In [ ]:
f = f'attack1w_results_{PIPELINE_VERSION}.pkl'
if os.path.exists(f):
    nb3c = joblib.load(f)['per_seed']; chk = []
    for sname in SETTINGS_TO_RUN:
        if sname not in nb3c[SEEDS[0]] or len(have(sname)) != len(SEEDS): continue
        full = np.mean([nb3c[s][sname]['per_class_evasion'][3:5] for s in SEEDS], 0); samp = np.mean([R[s][sname]['evasion_at'][25] for s in SEEDS], 0)
        chk.append({'setting': sname, 'NB3c full set, 25 rounds [ransomware, reverseshell]': np.round(full, 3).tolist(), 'this run, round 25 (3,000 rows each)': np.round(samp, 3).tolist()})
    print(pd.DataFrame(chk).to_string(index=False) if chk else 'no matching settings')
else: print('NB3c results file not found -- check skipped')

## 7. Figure and save

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8)); colors = {'cyber_0.3': 'tab:blue', 'cyber_0.5': 'tab:red', 'cyber_0.7': 'tab:green'}
for sname in SETTINGS_TO_RUN:
    if len(have(sname)) != len(SEEDS): continue
    L = min(len(TR[s][sname]) for s in SEEDS)
    for k, (c, ls) in enumerate(zip(cls, ['-', '--'])):
        y = np.mean([[t[f'evaded_c{4 + k}'] for t in TR[s][sname][:L]] for s in SEEDS], 0); ax[0].plot(np.arange(1, L + 1), y, ls, color=colors[sname], label=f'{c} @ {SETTINGS[sname][1]} sigma')
    gp = np.mean([[t['median_gap_unevaded'] for t in TR[s][sname][:L]] for s in SEEDS], 0); ax[1].plot(np.arange(1, L + 1), gp, color=colors[sname], label=f'{SETTINGS[sname][1]} sigma')
ax[0].set_xlabel('greedy rounds'); ax[0].set_ylabel('share evaded'); ax[0].set_title('Hard classes: evasion vs search rounds (white-box)'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[1].axhline(0, color='k', ls=':'); ax[1].set_xlabel('greedy rounds'); ax[1].set_ylabel('median margin gap of rows not yet evaded (>0 = benign wins)'); ax[1].set_title('Is the search still making progress?'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'attack1x_convergence_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'SETTINGS': SETTINGS, 'SETTINGS_TO_RUN': SETTINGS_TO_RUN, 'MAX_ITERS': MAX_ITERS, 'N_PER_CLASS': N_PER_CLASS, 'conv_table': conv_table,
             'per_seed': {s: dict(R[s]) for s in SEEDS}, 'trace': {s: dict(TR[s]) for s in SEEDS}, 'info': INFO}, f'attack1x_results_{PIPELINE_VERSION}.pkl')
conv_table.to_csv(f'attack1x_convergence_table_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB3d', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'max_iters': MAX_ITERS, 'n_per_class': N_PER_CLASS, 'xgboost': xgb.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb3d_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack1x_results_{PIPELINE_VERSION}.pkl', f'attack1x_convergence_table_{PIPELINE_VERSION}.csv', f'attack1x_convergence_{PIPELINE_VERSION}.png', f'nb3d_run_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)